# Desafío Profesional — Data Science
## Caso de Negocio: Cambio Climático
# Etapa 2: Limpieza y Transformación de Datos

---

**Objetivo:** Integrar los 5 datasets originales en datasets maestros listos para el modelado, aplicando un pipeline ETL (Extract, Transform, Load) completo y documentado.

**Contexto:** En la Etapa 1 exploramos cada dataset por separado y descubrimos varios problemas: encabezados inconsistentes, agrupaciones del Banco Mundial mezcladas con países, "Demanda MEM" mezclada con renovables, y nombres de países que difieren entre fuentes. Esta etapa resuelve todo eso y produce 5 datasets limpios:

| # | Dataset de salida | Descripción | Filas esperadas |
|---|-------------------|-------------|------------------|
| 1 | `dataset_global.csv` | País × año × variables (CO2, PBI, población, renovables) | ~1,200 |
| 2 | `dataset_argentina_anual.csv` | Argentina año × variables (factor emisión, demanda, generación por fuente) | ~9 |
| 3 | `dataset_argentina_mensual.csv` | Argentina mes × variables (demanda, generación renovable) | ~96 |
| 4 | `dataset_argentina_provincias.csv` | Provincia × año × fuente × generación (formato long) | ~500 |
| 5 | `dataset_argentina_provincias_wide.csv` | Provincia × año con columnas por fuente y porcentajes (formato wide, listo para clustering) | ~130 |

**Estructura del notebook:**
1. Carga de datos originales
2. Limpieza individual de cada dataset (Extract + Transform)
3. Integración en datasets maestros (Merge)
4. Validación y exploración de los datasets resultantes
5. Exportación (Load)
6. Documentación ETL

---
## 1. Configuración y carga de datos originales

Replicamos la carga validada en la Etapa 1 para que este notebook sea autosuficiente.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import os

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 50)
pd.set_option('display.float_format', '{:.2f}'.format)
plt.style.use('seaborn-v0_8-whitegrid')

# RUTA — AJUSTAR SEGÚN TU ENTORNO
PATH = '/mnt/user-data/uploads/'
OUTPUT_PATH = '/mnt/user-data/outputs/datos_limpios/'
os.makedirs(OUTPUT_PATH, exist_ok=True)

FILE_DS1 = PATH + 'Emisio_n_CO2_en_Argentina.xlsx'
FILE_DS2 = PATH + 'Estadísticas_energía_mundial.xlsx'
FILE_DS3 = PATH + 'PBI_per_cápita_-_datos_Banco_Mundial.xls'
FILE_DS4 = PATH + 'Población_mundial_-_dataset_Banco_Mundial.xlsx'
FILE_DS5 = PATH + 'Proyectos_renovables_en_Argentina.xlsx'

print('✅ Configuración lista')

In [ ]:
# ============================================================
# FUNCIONES DE CARGA (validadas en Etapa 1)
# ============================================================

def cargar_hoja_ds2(filepath, sheet_name):
    df = pd.read_excel(filepath, sheet_name=sheet_name, header=None)
    year_row = df.iloc[2]
    col_names = ['pais_region']
    year_cols = []
    for i in range(1, len(year_row)):
        try:
            year = int(float(year_row.iloc[i]))
            if 1900 < year < 2100:
                col_names.append(year)
                year_cols.append(year)
            else:
                col_names.append(f'extra_{i}')
        except (ValueError, TypeError):
            col_names.append(f'extra_{i}')
    df_data = df.iloc[3:].copy()
    df_data.columns = col_names[:len(df_data.columns)]
    df_data = df_data[[c for c in df_data.columns if not str(c).startswith('extra_')]]
    df_data = df_data.dropna(subset=['pais_region'])
    df_data = df_data[~df_data['pais_region'].str.contains('Source|Copyright|n.a.', na=False)]
    df_data = df_data.reset_index(drop=True)
    for col in year_cols:
        if col in df_data.columns:
            df_data[col] = pd.to_numeric(df_data[col], errors='coerce')
    return df_data, year_cols

def cargar_banco_mundial(filepath):
    df = pd.read_excel(filepath, header=None, skiprows=3)
    headers = df.iloc[0].tolist()
    df = df.iloc[1:].copy()
    clean_headers = []
    for h in headers:
        try:
            y = int(float(h))
            clean_headers.append(y if 1900 < y < 2100 else str(h))
        except (ValueError, TypeError):
            clean_headers.append(str(h) if pd.notna(h) else 'drop')
    df.columns = clean_headers
    df = df.drop(columns=['drop'], errors='ignore')
    df = df.dropna(subset=['Country Name']).reset_index(drop=True)
    year_cols = [c for c in df.columns if isinstance(c, int)]
    for y in year_cols:
        df[y] = pd.to_numeric(df[y], errors='coerce')
    return df, year_cols

AGRUPACIONES_BM = {
    'AFE','AFW','ARB','CEB','CSS','EAP','EAR','EAS','ECA','ECS',
    'EMU','EUU','FCS','HIC','HPC','IBD','IBT','IDA','IDB','IDX',
    'INX','LAC','LCN','LDC','LIC','LMC','LMY','LTE','MEA','MIC',
    'MNA','NAC','OED','OSS','PRE','PSS','PST','SAS','SSA','SSF',
    'SST','TEA','TEC','TLA','TMN','TSA','TSS','UMC','WLD'
}

AGRUPACIONES_DS2 = ['World','OECD','G7','BRICS','Europe','European Union',
                    'CIS','America','North America','Latin America','Asia',
                    'Pacific','Africa','Middle-East']

print('✅ Funciones de carga definidas')

In [ ]:
# ============================================================
# CARGA DE TODOS LOS DATASETS ORIGINALES
# ============================================================

# DS1
ds1 = pd.read_excel(FILE_DS1, sheet_name='FACTOR DE EMISION OM SIMPLE',
                     skiprows=5, usecols=[1,2,3],
                     names=['anio','factor_expost','factor_exante'])
ds1['anio'] = ds1['anio'].astype(int)
for col in ['factor_expost','factor_exante']:
    ds1[col] = pd.to_numeric(ds1[col], errors='coerce')

# DS2
HOJAS_DS2 = {
    'CO2 emissions from fuel combus': 'co2_MtCO2',
    'Share of renewables in electri': 'share_renewables',
    'Electricity production': 'electricity_TWh',
    'Total energy consumption': 'energy_Mtoe',
    'Share of wind and solar in ele': 'share_wind_solar',
}
ds2 = {}
ds2_years = None
for hoja, nombre in HOJAS_DS2.items():
    df, years = cargar_hoja_ds2(FILE_DS2, hoja)
    ds2[nombre] = df
    if ds2_years is None: ds2_years = years

# DS3 y DS4
ds3, ds3_years = cargar_banco_mundial(FILE_DS3)
ds4, ds4_years = cargar_banco_mundial(FILE_DS4)
ds3_paises = ds3[~ds3['Country Code'].isin(AGRUPACIONES_BM)].copy()
ds4_paises = ds4[~ds4['Country Code'].isin(AGRUPACIONES_BM)].copy()

# DS5
ds5_raw = pd.read_excel(FILE_DS5, header=None)
ds5 = ds5_raw.iloc[2:].copy()
ds5.columns = ds5_raw.iloc[1].tolist()
ds5 = ds5.rename(columns={
    'AÑO':'anio','CENTRAL':'central_cod','CENTRAL DESCRIPCIÓN':'central_desc',
    'MAQUINA':'maquina','FUENTE DE ENERGÍA':'fuente_energia',
    'REGIÓN':'region','PROVINCIA':'provincia','MES':'mes',
    'ENERGÍA GENERADA [GWh]':'energia_gwh','Nueva Generación':'nueva_generacion'
})
ds5['anio'] = pd.to_numeric(ds5['anio'], errors='coerce').astype('Int64')
ds5['energia_gwh'] = pd.to_numeric(ds5['energia_gwh'], errors='coerce')
ds5['mes'] = pd.to_datetime(ds5['mes'], errors='coerce')
ds5 = ds5.dropna(axis=1, how='all')

# Separar MEM de renovable
ds5_renovable = ds5[ds5['fuente_energia'].str.strip() != 'Demanda MEM'].copy()
ds5_mem = ds5[ds5['fuente_energia'].str.strip() == 'Demanda MEM'].copy()

print('✅ Todos los datasets cargados')
print(f'   DS1: {ds1.shape} | DS2: {len(ds2)} hojas | DS3: {ds3_paises.shape} | DS4: {ds4_paises.shape}')
print(f'   DS5 renovable: {ds5_renovable.shape} | DS5 MEM: {ds5_mem.shape}')

---
## 2. Limpieza y transformación (Transform)

### 2.1 — Pivoteo de DS2 a formato long

Los datos de DS2 están en formato "wide" (un año por columna). Para poder hacer joins con DS3/DS4 y para facilitar el modelado, necesitamos pasarlos a formato "long" (una fila por país-año).

En términos simples: imaginate una planilla de Excel donde cada columna es un año (1990, 1991, ..., 2018). Lo que hacemos es "aplanarla" para que en vez de 29 columnas de años, tengamos una sola columna `anio` y una columna `valor`. Esto multiplica las filas pero simplifica enormemente los joins y los análisis.

In [ ]:
# Pivotear cada hoja de DS2 a formato long y luego mergear
ds2_long_list = []

for nombre, df in ds2.items():
    # Filtrar solo países individuales (no regiones)
    df_paises = df[~df['pais_region'].isin(AGRUPACIONES_DS2)].copy()
    
    # Melt: de wide a long
    df_melted = df_paises.melt(
        id_vars=['pais_region'],
        value_vars=ds2_years,
        var_name='anio',
        value_name=nombre
    )
    df_melted['anio'] = df_melted['anio'].astype(int)
    ds2_long_list.append(df_melted)

# Merge todas las variables de DS2 en un solo DataFrame
ds2_long = ds2_long_list[0]
for df_next in ds2_long_list[1:]:
    ds2_long = ds2_long.merge(df_next, on=['pais_region', 'anio'], how='outer')

print(f'DS2 en formato long: {ds2_long.shape}')
print(f'Países: {ds2_long["pais_region"].nunique()} | Años: {ds2_long["anio"].min()}-{ds2_long["anio"].max()}')
print(f'\nColumnas: {list(ds2_long.columns)}')
print(f'\nPrimeras filas:')
display(ds2_long.head())
print(f'\nNulos por columna:')
display(ds2_long.isnull().sum())

### 2.2 — Pivoteo de DS3 y DS4 a formato long

Mismo proceso para los datos del Banco Mundial. Además, creamos el **mapeo de nombres** entre DS2 (inglés) y DS3/DS4 (nombres del Banco Mundial, muchos en español).

In [ ]:
# Mapeo de nombres DS2 → Banco Mundial
# Construido manualmente verificando las diferencias
MAPEO_DS2_A_BM = {
    'United States': 'Estados Unidos',
    'United Kingdom': 'Reino Unido',
    'South Korea': 'Corea, República de',
    'Russia': 'Federación de Rusia',
    'Iran': 'Irán, República Islámica del',
    'Venezuela': 'Venezuela, RB',
    'Egypt': 'Egipto, República Árabe de',
    'Czech Republic': 'Chequia',
    'Turkey': 'Türkiye',
    'Taiwan': None,  # No está en Banco Mundial
}

# Mapeo inverso para obtener country_code
bm_to_code = ds3_paises.set_index('Country Name')['Country Code'].to_dict()

# Función para obtener el código ISO de un país de DS2
def get_country_code(pais_ds2):
    pais_bm = MAPEO_DS2_A_BM.get(pais_ds2, pais_ds2)
    if pais_bm is None:
        return None
    return bm_to_code.get(pais_bm, None)

# Verificar mapeo
paises_ds2 = ds2_long['pais_region'].unique()
mapeados = 0
no_mapeados = []
for p in paises_ds2:
    code = get_country_code(p)
    if code:
        mapeados += 1
    else:
        no_mapeados.append(p)

print(f'Mapeo DS2 → Banco Mundial:')
print(f'  Mapeados: {mapeados}/{len(paises_ds2)}')
if no_mapeados:
    print(f'  No mapeados: {no_mapeados}')
    print(f'  → Estos países se excluyen del dataset global integrado')

In [ ]:
# Pivotear DS3 (PBI) a formato long — solo países reales
years_range = [y for y in ds3_years if 1990 <= y <= 2018]

ds3_long = ds3_paises.melt(
    id_vars=['Country Name', 'Country Code'],
    value_vars=years_range,
    var_name='anio',
    value_name='pbi_per_capita'
)
ds3_long['anio'] = ds3_long['anio'].astype(int)

# Pivotear DS4 (Población) a formato long
ds4_long = ds4_paises.melt(
    id_vars=['Country Name', 'Country Code'],
    value_vars=years_range,
    var_name='anio',
    value_name='poblacion'
)
ds4_long['anio'] = ds4_long['anio'].astype(int)

print(f'DS3 long (PBI): {ds3_long.shape}')
print(f'DS4 long (Población): {ds4_long.shape}')

---
## 3. Integración de datasets

### 3.1 — Dataset maestro global

Ahora viene la parte central de la Etapa 2: **unir todo en un solo dataset**. El proceso es:
1. Agregar `country_code` a DS2 usando el mapeo de nombres.
2. Hacer JOIN con DS3 (PBI) y DS4 (Población) por `country_code` + `anio`.
3. Calcular variables derivadas: CO2 per cápita, intensidad carbónica, eficiencia energética.

Pensalo como armar un rompecabezas: cada dataset tiene una pieza de la historia (emisiones, riqueza, población, renovables), y el join los une en una sola tabla donde cada fila es un país en un año con todas sus variables.

In [ ]:
# Agregar country_code a DS2 long
ds2_long['country_code'] = ds2_long['pais_region'].apply(get_country_code)

# Eliminar países sin mapeo (ej: Taiwan)
ds2_con_code = ds2_long.dropna(subset=['country_code']).copy()
print(f'DS2 con country_code: {ds2_con_code.shape} ({ds2_con_code["pais_region"].nunique()} países)')

# JOIN con DS3 (PBI)
global_df = ds2_con_code.merge(
    ds3_long[['Country Code', 'anio', 'pbi_per_capita']],
    left_on=['country_code', 'anio'],
    right_on=['Country Code', 'anio'],
    how='left'
).drop(columns=['Country Code'])

# JOIN con DS4 (Población)
global_df = global_df.merge(
    ds4_long[['Country Code', 'anio', 'poblacion']],
    left_on=['country_code', 'anio'],
    right_on=['Country Code', 'anio'],
    how='left'
).drop(columns=['Country Code'])

print(f'\nDataset global tras merge: {global_df.shape}')
print(f'Nulos por columna:')
display(global_df.isnull().sum())

In [ ]:
# Calcular variables derivadas
global_df['co2_per_capita'] = np.where(
    global_df['poblacion'] > 0,
    (global_df['co2_MtCO2'] * 1e6) / global_df['poblacion'],  # MtCO2 → tCO2, / personas
    np.nan
)

global_df['co2_per_pbi'] = np.where(
    global_df['pbi_per_capita'] > 0,
    global_df['co2_per_capita'] / global_df['pbi_per_capita'] * 1000,  # kgCO2 por US$ de PBI
    np.nan
)

global_df['co2_per_energy'] = np.where(
    global_df['energy_Mtoe'] > 0,
    global_df['co2_MtCO2'] / global_df['energy_Mtoe'],  # MtCO2 / Mtoe
    np.nan
)

# Renombrar columna pais para claridad
global_df = global_df.rename(columns={'pais_region': 'pais'})

# Ordenar
global_df = global_df.sort_values(['pais', 'anio']).reset_index(drop=True)

# Reordenar columnas
col_order = ['pais', 'country_code', 'anio', 'co2_MtCO2', 'poblacion', 'pbi_per_capita',
             'co2_per_capita', 'electricity_TWh', 'energy_Mtoe',
             'share_renewables', 'share_wind_solar', 'co2_per_pbi', 'co2_per_energy']
global_df = global_df[col_order]

print('✅ Dataset maestro global construido')
print(f'   Shape: {global_df.shape}')
print(f'   Países: {global_df["pais"].nunique()}')
print(f'   Período: {global_df["anio"].min()}-{global_df["anio"].max()}')
print(f'   Columnas: {list(global_df.columns)}')
print(f'\nPrimeras filas (Argentina):')
display(global_df[global_df['pais']=='Argentina'].head())
print(f'\nEstadísticas descriptivas:')
display(global_df.describe())

### 3.2 — Dataset Argentina anual

Integramos DS1 (factor emisión), DS5 (generación renovable y demanda MEM), y las variables de Argentina del dataset global. El desglose por fuente permite que en la Etapa 3 podamos modelar cuánto contribuye cada tipo de renovable a la reducción del factor de emisión.

In [ ]:
# Generación renovable anual por fuente
gen_fuente = ds5_renovable.groupby(['anio', 'fuente_energia'])['energia_gwh'].sum().reset_index()
gen_pivot = gen_fuente.pivot_table(index='anio', columns='fuente_energia', values='energia_gwh', aggfunc='sum').fillna(0)
gen_pivot.columns = ['gen_' + c.strip().lower().replace(' ', '_').replace('<=', 'lte') + '_GWh' for c in gen_pivot.columns]
gen_pivot['gen_renovable_total_GWh'] = gen_pivot.sum(axis=1)
gen_pivot = gen_pivot.reset_index()
gen_pivot['anio'] = gen_pivot['anio'].astype(int)

# Demanda MEM anual
mem_anual = ds5_mem.groupby('anio')['energia_gwh'].sum().reset_index()
mem_anual.columns = ['anio', 'demanda_MEM_GWh']
mem_anual['anio'] = mem_anual['anio'].astype(int)

# Merge: generación + demanda
arg_anual = gen_pivot.merge(mem_anual, on='anio', how='outer')

# Merge: factor de emisión (DS1)
arg_anual = arg_anual.merge(ds1, on='anio', how='outer')

# Merge: variables macro de Argentina del dataset global
arg_global = global_df[global_df['pais'] == 'Argentina'][['anio', 'co2_MtCO2', 'pbi_per_capita', 'poblacion', 'share_renewables']].copy()
arg_global.columns = ['anio', 'co2_arg_MtCO2', 'pbi_arg', 'poblacion_arg', 'share_renewables_arg']
arg_anual = arg_anual.merge(arg_global, on='anio', how='outer')

# Ratio renovable/demanda
arg_anual['ratio_renovable_pct'] = np.where(
    arg_anual['demanda_MEM_GWh'] > 0,
    (arg_anual['gen_renovable_total_GWh'] / arg_anual['demanda_MEM_GWh']) * 100,
    np.nan
)

# Ordenar por año
arg_anual = arg_anual.sort_values('anio').reset_index(drop=True)

print('✅ Dataset Argentina anual construido')
print(f'   Shape: {arg_anual.shape}')
print(f'   Período: {arg_anual["anio"].min()}-{arg_anual["anio"].max()}')
print(f'   Columnas: {list(arg_anual.columns)}')
print(f'\nDatos completos:')
display(arg_anual)

### 3.3 — Dataset Argentina mensual

Este dataset tiene la granularidad mensual necesaria para los modelos de series temporales (ARIMA/Prophet en Etapa 3) y para las redes neuronales recurrentes (LSTM/GRU en Etapa 4). La granularidad mensual permite capturar estacionalidad y tendencias que no se ven a nivel anual.

Excluimos 2019 porque tiene datos incompletos (solo hasta cierto mes).

In [ ]:
# Generación renovable mensual por fuente
ds5_renov_clean = ds5_renovable[ds5_renovable['anio'] <= 2018].copy()
ds5_renov_clean['mes_dt'] = ds5_renov_clean['mes'].dt.to_period('M').dt.to_timestamp()

gen_mensual = ds5_renov_clean.groupby(['mes_dt', 'fuente_energia'])['energia_gwh'].sum().reset_index()
gen_mensual_pivot = gen_mensual.pivot_table(
    index='mes_dt', columns='fuente_energia', values='energia_gwh', aggfunc='sum'
).fillna(0)
gen_mensual_pivot.columns = ['gen_' + c.strip().lower().replace(' ','_').replace('<=','lte') + '_GWh' for c in gen_mensual_pivot.columns]
gen_mensual_pivot['gen_renovable_total_GWh'] = gen_mensual_pivot.sum(axis=1)
gen_mensual_pivot = gen_mensual_pivot.reset_index().rename(columns={'mes_dt': 'mes'})

# Demanda MEM mensual
ds5_mem_clean = ds5_mem[ds5_mem['anio'] <= 2018].copy()
ds5_mem_clean['mes_dt'] = ds5_mem_clean['mes'].dt.to_period('M').dt.to_timestamp()
mem_mensual_df = ds5_mem_clean.groupby('mes_dt')['energia_gwh'].sum().reset_index()
mem_mensual_df.columns = ['mes', 'demanda_MEM_GWh']

# Merge
arg_mensual = gen_mensual_pivot.merge(mem_mensual_df, on='mes', how='outer')
arg_mensual['ratio_renovable_pct'] = np.where(
    arg_mensual['demanda_MEM_GWh'] > 0,
    (arg_mensual['gen_renovable_total_GWh'] / arg_mensual['demanda_MEM_GWh']) * 100,
    np.nan
)
arg_mensual = arg_mensual.sort_values('mes').reset_index(drop=True)

print('✅ Dataset Argentina mensual construido')
print(f'   Shape: {arg_mensual.shape}')
print(f'   Período: {arg_mensual["mes"].min()} a {arg_mensual["mes"].max()}')
print(f'   Columnas: {list(arg_mensual.columns)}')
print(f'\nPrimeras y últimas filas:')
display(arg_mensual.head())
display(arg_mensual.tail())

### 3.4 — Dataset Argentina por provincia

El mapa energético renovable de Argentina tiene una fuerte dimensión geográfica: la Patagonia es eólica, Cuyo es hidro+solar, el Noroeste es biomasa+hidro. Este dataset permite explorar esas diferencias y hacer clustering de provincias por perfil energético en la Etapa 3.

Cada fila representa una provincia en un año con una fuente de energía específica y su generación en GWh.

In [ ]:
# Dataset por provincia, año y fuente
arg_prov = ds5_renovable[ds5_renovable['anio'] <= 2018].groupby(
    ['anio', 'provincia', 'region', 'fuente_energia']
)['energia_gwh'].sum().reset_index()

arg_prov['anio'] = arg_prov['anio'].astype(int)
arg_prov = arg_prov.sort_values(['provincia', 'anio', 'fuente_energia']).reset_index(drop=True)

# También crear versión pivoteada por fuente para clustering
arg_prov_wide = arg_prov.groupby(['anio', 'provincia']).apply(
    lambda g: pd.Series({
        'gen_total_GWh': g['energia_gwh'].sum(),
        'gen_eolico_GWh': g[g['fuente_energia']=='EOLICO']['energia_gwh'].sum(),
        'gen_solar_GWh': g[g['fuente_energia']=='SOLAR']['energia_gwh'].sum(),
        'gen_hidro_GWh': g[g['fuente_energia']=='HIDRO <=50MW']['energia_gwh'].sum(),
        'gen_biomasa_GWh': g[g['fuente_energia'].isin(['BIOMASA','BIOGAS','BIODIESEL'])]['energia_gwh'].sum(),
    })
).reset_index()

# Porcentaje de cada fuente
for col in ['gen_eolico_GWh', 'gen_solar_GWh', 'gen_hidro_GWh', 'gen_biomasa_GWh']:
    pct_col = col.replace('_GWh', '_pct')
    arg_prov_wide[pct_col] = np.where(
        arg_prov_wide['gen_total_GWh'] > 0,
        arg_prov_wide[col] / arg_prov_wide['gen_total_GWh'] * 100,
        0
    )

print('✅ Dataset Argentina por provincia construido')
print(f'   Versión long: {arg_prov.shape} | Versión wide: {arg_prov_wide.shape}')
print(f'   Provincias: {arg_prov["provincia"].nunique()}')
print(f'   Período: {arg_prov["anio"].min()}-{arg_prov["anio"].max()}')
print(f'\nEjemplo — 2018, top 5 provincias:')
display(arg_prov_wide[arg_prov_wide['anio']==2018].sort_values('gen_total_GWh', ascending=False).head())

---
## 4. Validación de los datasets resultantes

Antes de exportar, verificamos que los datos sean consistentes: sin duplicados, con los tipos correctos, y que los valores derivados tengan sentido.

In [ ]:
print('=' * 70)
print('VALIDACIÓN DE DATASETS')
print('=' * 70)

for nombre, df in [('Global', global_df), ('Argentina anual', arg_anual), 
                    ('Argentina mensual', arg_mensual), ('Argentina provincias', arg_prov_wide)]:
    print(f'\n--- {nombre} ---')
    print(f'  Shape: {df.shape}')
    print(f'  Duplicados: {df.duplicated().sum()}')
    print(f'  Nulos totales: {df.isnull().sum().sum()} ({df.isnull().sum().sum()/(df.shape[0]*df.shape[1])*100:.1f}%)')
    print(f'  Tipos: {dict(df.dtypes.value_counts())}')

# Validaciones cruzadas
print(f'\n--- Validaciones cruzadas ---')
# CO2 per cápita de Argentina debe ser razonable (3-5 tCO2/persona)
arg_co2pc = global_df[global_df['pais']=='Argentina']['co2_per_capita'].dropna()
print(f'  CO2 per cápita Argentina: {arg_co2pc.min():.1f} - {arg_co2pc.max():.1f} tCO2/persona ✅' if arg_co2pc.between(1, 10).all() else '  ⚠️ CO2 per cápita fuera de rango')

# Ratio renovable debe ser < 100%
ratio_ok = arg_anual['ratio_renovable_pct'].dropna().between(0, 100).all()
print(f'  Ratio renovable Argentina: 0-100% ✅' if ratio_ok else '  ⚠️ Ratio renovable fuera de rango')

# Dataset mensual debe tener 12 meses por año (excepto extremos)
meses_por_anio = arg_mensual.groupby(arg_mensual['mes'].dt.year).size()
print(f'  Meses por año en dataset mensual: {dict(meses_por_anio)}')

### Visualización rápida de validación

Un vistazo rápido a los datos integrados para confirmar que todo tiene sentido visualmente.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# 1. Nulos en dataset global
ax = axes[0, 0]
nulos = global_df.isnull().sum().sort_values(ascending=True)
nulos.plot(kind='barh', ax=ax, color='#2E86C1')
ax.set_title('Nulos por columna — Dataset global')
ax.set_xlabel('Cantidad de nulos')

# 2. CO2 per cápita de Argentina en el dataset global
ax = axes[0, 1]
arg_data = global_df[global_df['pais'] == 'Argentina']
ax.plot(arg_data['anio'], arg_data['co2_per_capita'], 'o-', color='#75AADB', linewidth=2)
ax.set_title('Argentina — CO2 per cápita (dataset global)')
ax.set_xlabel('Año'); ax.set_ylabel('tCO2/persona')

# 3. Dataset Argentina anual — ratio renovable
ax = axes[1, 0]
arg_plot = arg_anual.dropna(subset=['ratio_renovable_pct'])
ax.bar(arg_plot['anio'], arg_plot['ratio_renovable_pct'], color='#27AE60')
ax.set_title('Argentina — Ratio renovable/demanda (dataset anual)')
ax.set_xlabel('Año'); ax.set_ylabel('%')

# 4. Dataset Argentina mensual — generación renovable
ax = axes[1, 1]
ax.plot(arg_mensual['mes'], arg_mensual['gen_renovable_total_GWh'], color='#27AE60', linewidth=1.5)
ax.set_title('Argentina — Generación renovable mensual (dataset mensual)')
ax.set_xlabel('Mes'); ax.set_ylabel('GWh')
ax.tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

---
## 5. Exportación (Load)

Guardamos los 4 datasets como CSV para que estén listos para las Etapas 3 y 4.

In [ ]:
# Exportar
global_df.to_csv(OUTPUT_PATH + 'dataset_global.csv', index=False)
arg_anual.to_csv(OUTPUT_PATH + 'dataset_argentina_anual.csv', index=False)
arg_mensual.to_csv(OUTPUT_PATH + 'dataset_argentina_mensual.csv', index=False)
arg_prov.to_csv(OUTPUT_PATH + 'dataset_argentina_provincias.csv', index=False)
arg_prov_wide.to_csv(OUTPUT_PATH + 'dataset_argentina_provincias_wide.csv', index=False)

print('✅ Datasets exportados a', OUTPUT_PATH)
print()
for f in os.listdir(OUTPUT_PATH):
    size = os.path.getsize(OUTPUT_PATH + f) / 1024
    print(f'  {f:<45s} {size:>8.1f} KB')

---
## 6. Documentación del proceso ETL

### Extract (Extracción)
- **DS1** (Emisiones CO2 Argentina): Excel con 3 hojas. Se extrajo la hoja 'FACTOR DE EMISION OM SIMPLE', saltando 5 filas de metadata. Columnas: año, factor ex-post, factor ex-ante.
- **DS2** (Energía mundial): Excel con 27 hojas. Se extrajeron 5 hojas relevantes usando `cargar_hoja_ds2()`. Cada hoja tiene 3 filas de encabezado no estándar.
- **DS3** (PBI per cápita): Excel del Banco Mundial con 3 filas de metadata. Se usó `cargar_banco_mundial()`. 264 entidades, 60 años.
- **DS4** (Población mundial): Mismo formato que DS3. Misma función de carga.
- **DS5** (Renovables Argentina): Excel con fila 0 vacía y headers en fila 1. 7,513 registros incluyendo 'Demanda MEM'.

### Transform (Transformación)

**Limpieza:**
- Se corrigieron encabezados no estándar en los 5 datasets.
- Se separaron 47 agrupaciones del Banco Mundial de los 217 países reales (`AGRUPACIONES_BM`).
- Se separó 'Demanda MEM' de la generación renovable en DS5.
- Se excluyó 2019 de DS5 por datos incompletos.
- Se creó mapeo de nombres de países DS2 → Banco Mundial (`MAPEO_DS2_A_BM`). Taiwan se excluyó por no estar en BM.

**Transformación:**
- DS2, DS3, DS4: pivoteados de formato wide (año-columna) a long (año-fila) con `melt()`.
- DS5 renovable: agregado a nivel anual por fuente y a nivel mensual.
- DS5 provincias: agregado por provincia, año y fuente; versión wide con porcentajes por fuente.

**Variables derivadas:**
- `co2_per_capita`: (CO2 MtCO2 × 10⁶) / población → tCO2/persona
- `co2_per_pbi`: CO2 per cápita / PBI per cápita × 1000 → kgCO2/US$
- `co2_per_energy`: CO2 MtCO2 / Energía Mtoe → intensidad carbónica
- `ratio_renovable_pct`: generación renovable / demanda MEM × 100
- Porcentajes por fuente en dataset provincias.

**Integración:**
- Dataset global: JOIN de DS2 + DS3 + DS4 por `country_code` + `anio`.
- Dataset Argentina anual: MERGE de generación renovable + demanda MEM + factor emisión + variables macro.
- Dataset Argentina mensual: generación mensual + demanda mensual, período 2011-2018.
- Dataset Argentina provincias: generación por provincia × año × fuente.

### Load (Carga)
- 5 archivos CSV exportados al directorio `datos_limpios/`: dataset global, Argentina anual, Argentina mensual, Argentina provincias (long) y Argentina provincias (wide para clustering).
- Formatos: UTF-8, separador coma, sin índice.
- Listos para ser cargados directamente con `pd.read_csv()` en las Etapas 3 y 4.